# BudgetBuddy Multimodal Capabilities Notebook

This notebook validates **existing multimodal capabilities** in BudgetBuddy and documents practical prompt strategies.

## Goals
- Integrate and validate multimodal capabilities relevant to the project use case.
- Implement and test image understanding and document parsing workflows.
- Add a multimodal search/interaction flow across text + image-derived content.
- Measure cross-modal consistency and relevance.
- Document modality-specific prompt design strategies and trade-offs.

> Expected backend: `http://localhost:8000` with the BudgetBuddy API running.

## 1) Environment Setup and Dependency Checks

Install/import required packages, verify compute availability, define project paths, and centralize runtime configuration for multimodal inference/evaluation.

In [1]:
from pathlib import Path
from datetime import datetime
import os
import json
import time
import statistics
import platform
import requests

# Runtime configuration
API_URL = os.getenv("BUDGETBUDDY_API_URL", "http://localhost:8000")
USERNAME = os.getenv("BUDGETBUDDY_TEST_USER", "lkona")
TOP_K = int(os.getenv("BUDGETBUDDY_TOP_K", "3"))
AMOUNT_TOLERANCE = float(os.getenv("BUDGETBUDDY_AMOUNT_TOLERANCE", "2.0"))
SEED = int(os.getenv("BUDGETBUDDY_SEED", "42"))

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "docs").exists() is False and (PROJECT_ROOT.parent / "docs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RECEIPT_DIR = PROJECT_ROOT / "tests" / "fixtures" / "receipts_new"

print("Environment")
print("- Python:", platform.python_version())
print("- Platform:", platform.platform())
print("- API_URL:", API_URL)
print("- USERNAME:", USERNAME)
print("- PROJECT_ROOT:", PROJECT_ROOT)
print("- RECEIPT_DIR:", RECEIPT_DIR)
print("- RECEIPT_DIR exists:", RECEIPT_DIR.exists())

# Optional GPU check
try:
    import torch
    print("- Torch CUDA available:", torch.cuda.is_available())
except Exception:
    print("- Torch not installed (GPU check skipped)")

# Optional dependency checks for document and embedding utilities
for pkg, import_name in [("pypdf", "pypdf"), ("Pillow", "PIL"), ("scikit-learn", "sklearn")]:
    try:
        __import__(import_name)
        print(f"- {pkg}: available")
    except Exception:
        print(f"- {pkg}: not installed (fallback paths will be used)")


def login_get_token(api_url: str = API_URL, username: str = USERNAME) -> str:
    """Authenticate and return JWT token."""
    response = requests.post(f"{api_url}/api/auth/login", json={"username": username}, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"Login failed ({response.status_code}): {response.text}")
    token = response.json().get("token")
    if not token:
        raise RuntimeError("Login response missing token")
    return token

TOKEN = login_get_token()
print("Login OK. Token prefix:", TOKEN[:16], "...")

Environment
- Python: 3.12.2
- Platform: Windows-11-10.0.26200-SP0
- API_URL: http://localhost:8000
- USERNAME: lkona
- PROJECT_ROOT: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy
- RECEIPT_DIR: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\tests\fixtures\receipts_new
- RECEIPT_DIR exists: True
- Torch not installed (GPU check skipped)
- pypdf: available
- Pillow: available
- scikit-learn: not installed (fallback paths will be used)
Login OK. Token prefix: eyJhbGciOiJIUzI1 ...


## 2) Load Multimodal Models and Utilities

Initialize modality-aware helpers for:
- Vision/document parsing through `/api/parse-receipt`
- Text understanding through `/api/parse-expense`
- Optional embedding and retrieval helpers for multimodal search

In [2]:
def parse_expense_text(text: str, token: str = TOKEN, api_url: str = API_URL):
    """Parse natural-language expense text using backend LLM pipeline."""
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    response = requests.post(
        f"{api_url}/api/parse-expense",
        headers=headers,
        json={"text": text},
        timeout=60,
    )
    if response.status_code != 200:
        raise RuntimeError(f"parse-expense failed ({response.status_code}): {response.text}")
    return response.json()


def parse_receipt_image(image_path: Path, token: str = TOKEN, api_url: str = API_URL):
    """Parse receipt image via multimodal receipt endpoint."""
    if not image_path.exists():
        raise FileNotFoundError(f"Missing image: {image_path}")

    headers = {"Authorization": f"Bearer {token}"}
    mime_type = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"

    with image_path.open("rb") as file_obj:
        files = {"file": (image_path.name, file_obj, mime_type)}
        response = requests.post(
            f"{api_url}/api/parse-receipt",
            headers=headers,
            files=files,
            timeout=90,
        )

    if response.status_code != 200:
        raise RuntimeError(f"parse-receipt failed ({response.status_code}): {response.text}")
    return response.json()


def extract_core_fields(response_json: dict) -> dict:
    """Normalize fields from API responses to a common schema."""
    payload = response_json.get("parsed_data", response_json)
    return {
        "amount": payload.get("amount"),
        "category": payload.get("category"),
        "description": payload.get("description"),
        "date": payload.get("date") or payload.get("expense_date"),
    }


def image_to_text_statement(image_fields: dict) -> str:
    """Convert image-derived parse output into a text statement for cross-modal checks."""
    amount = image_fields.get("amount", 0)
    category = image_fields.get("category") or "Other"
    description = image_fields.get("description") or category.lower()
    date_value = image_fields.get("date") or datetime.now().strftime("%Y-%m-%d")
    return f"I spent ${amount} on {description} in {category} on {date_value}"

## 3) Image Understanding Pipeline

Run receipt images through the backend vision + OCR parsing path and return structured fields with lightweight confidence metadata (field completeness proxy).

In [3]:
sample_images = [
    RECEIPT_DIR / "sample_receipt_1.jpg",
    RECEIPT_DIR / "sample_receipt_2.png",
    RECEIPT_DIR / "starbucks_receipt.jpg",
]

image_results = []

for image_path in sample_images:
    try:
        raw = parse_receipt_image(image_path)
        fields = extract_core_fields(raw)
        completeness = sum(v is not None and str(v).strip() != "" for v in fields.values()) / 4

        row = {
            "modality": "image",
            "source": image_path.name,
            "fields": fields,
            "confidence_proxy": round(completeness, 2),
            "raw": raw,
        }
        image_results.append(row)
        print(f"✓ {image_path.name}: {fields} | confidence_proxy={row['confidence_proxy']}")
    except Exception as exc:
        print(f"✗ {image_path.name}: {exc}")

print(f"Image pipeline completed. Successes: {len(image_results)}/{len(sample_images)}")

✗ sample_receipt_1.jpg: Missing image: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\tests\fixtures\receipts_new\sample_receipt_1.jpg
✗ sample_receipt_2.png: Missing image: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\tests\fixtures\receipts_new\sample_receipt_2.png
✗ starbucks_receipt.jpg: Missing image: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\tests\fixtures\receipts_new\starbucks_receipt.jpg
Image pipeline completed. Successes: 0/3


## 4) Document Parsing Pipeline (PDF/Image/OCR)

Treat receipts as document-like inputs (scanned images), normalize extracted content, and optionally parse PDF files when present.

In [4]:
def normalize_document_record(source_name: str, fields: dict) -> dict:
    text_blob = " ".join(str(v) for v in fields.values() if v is not None)
    return {
        "modality": "document",
        "source": source_name,
        "text": text_blob.strip(),
        "fields": fields,
        "metadata": {
            "parsed_at": datetime.utcnow().isoformat(),
            "source_type": "receipt-image-ocr",
        },
    }


document_records = [normalize_document_record(item["source"], item["fields"]) for item in image_results]
print(f"Document-like records from OCR receipts: {len(document_records)}")

# Optional PDF extraction path
pdf_files = list((PROJECT_ROOT / "tests" / "fixtures" / "receipts_new").rglob("*.pdf")) if (PROJECT_ROOT / "tests" / "fixtures" / "receipts_new").exists() else []
if not pdf_files:
    print("No PDFs found under tests/fixtures/receipts_new. Skipping PDF extraction.")
else:
    try:
        from pypdf import PdfReader
        for pdf_path in pdf_files[:3]:
            reader = PdfReader(str(pdf_path))
            extracted = "\n".join((page.extract_text() or "") for page in reader.pages)
            document_records.append({
                "modality": "document",
                "source": pdf_path.name,
                "text": extracted[:4000],
                "fields": {"description": extracted[:200]},
                "metadata": {"source_type": "pdf", "parsed_at": datetime.utcnow().isoformat()},
            })
        print(f"Added PDF records: {len(pdf_files[:3])}")
    except Exception as exc:
        print("PDF parsing skipped due to missing dependency or parse issue:", exc)

Document-like records from OCR receipts: 0
Added PDF records: 1


C:\Users\jyoth\AppData\Local\Temp\ipykernel_27968\3315428499.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "metadata": {"source_type": "pdf", "parsed_at": datetime.utcnow().isoformat()},


## 4b) PDF → Parse → Save to Database

For every PDF found in `tests/fixtures/receipts`, extract text with `pypdf`, send it through `/api/parse-expense` to get structured fields, then save as an expense via `POST /api/expenses` — exactly matching the app's upload-and-save flow.

In [5]:
VALID_CATEGORIES = {"Food", "Transportation", "Entertainment", "Shopping", "Bills", "Healthcare", "Education", "Other"}

def _normalize_category(cat: str) -> str:
    """Map raw category string to the closest valid backend category."""
    if not cat:
        return "Other"
    title = cat.strip().title()
    if title in VALID_CATEGORIES:
        return title
    lower = cat.strip().lower()
    mapping = {
        "food": "Food", "grocery": "Food", "groceries": "Food", "restaurant": "Food",
        "coffee": "Food", "drink": "Food", "dining": "Food",
        "transport": "Transportation", "uber": "Transportation", "lyft": "Transportation",
        "gas": "Transportation", "fuel": "Transportation", "transit": "Transportation",
        "entertainment": "Entertainment", "movie": "Entertainment", "cinema": "Entertainment",
        "shop": "Shopping", "shopping": "Shopping", "retail": "Shopping",
        "bill": "Bills", "bills": "Bills", "utility": "Bills", "utilities": "Bills",
        "health": "Healthcare", "healthcare": "Healthcare", "pharmacy": "Healthcare",
        "education": "Education", "school": "Education", "tuition": "Education",
    }
    for key, val in mapping.items():
        if key in lower:
            return val
    return "Other"


def save_expense_to_db(fields: dict, token: str = TOKEN, api_url: str = API_URL) -> dict:
    """POST parsed expense fields to /api/expenses (creates a DB record)."""
    amount = fields.get("amount")
    if amount is None or float(amount) <= 0:
        raise ValueError(f"Invalid amount: {amount}")

    payload = {
        "amount": float(amount),
        "category": _normalize_category(fields.get("category", "")),
        "description": (fields.get("description") or "")[:200],
        "date": fields.get("date") or datetime.now().strftime("%Y-%m-%d"),
    }

    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    response = requests.post(f"{api_url}/api/expenses", headers=headers, json=payload, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"Save failed ({response.status_code}): {response.text}")
    return response.json()


def pdf_text_to_expense(pdf_path: Path, token: str = TOKEN, api_url: str = API_URL) -> dict:
    """Extract text from a PDF, parse it via LLM, and return structured expense fields."""
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError("pypdf not installed. Run: pip install pypdf")

    reader = PdfReader(str(pdf_path))
    text = "\n".join((page.extract_text() or "") for page in reader.pages).strip()
    if not text:
        raise ValueError(f"No extractable text in {pdf_path.name}")

    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    parse_response = requests.post(
        f"{api_url}/api/parse-expense",
        headers=headers,
        json={"text": text[:500]},
        timeout=60,
    )
    if parse_response.status_code != 200:
        raise RuntimeError(f"parse-expense failed ({parse_response.status_code}): {parse_response.text}")

    return extract_core_fields(parse_response.json())


pdf_dir = PROJECT_ROOT / "tests" / "fixtures" / "receipts_new"
pdf_files = list(pdf_dir.rglob("*.pdf")) if pdf_dir.exists() else []

if not pdf_files:
    print(f"No PDF files found in {pdf_dir}")
    print("(Place .pdf receipts there to test this flow)")
    pdf_saved_records = []
else:
    pdf_saved_records = []
    print(f"Found {len(pdf_files)} PDF(s) in {pdf_dir}\n")

    for pdf_path in pdf_files:
        print(f"Processing: {pdf_path.name}")
        try:
            fields = pdf_text_to_expense(pdf_path)
            print(f"  Parsed fields: {fields}")
            saved = save_expense_to_db(fields)
            print(f"  ✓ Saved  →  id={saved.get('id')}  amount={saved.get('amount')}  category={saved.get('category')}  date={saved.get('expense_date')}")
            pdf_saved_records.append({"pdf": pdf_path.name, "saved": saved})
        except Exception as exc:
            print(f"  ✗ Failed: {exc}")
            pdf_saved_records.append({"pdf": pdf_path.name, "error": str(exc)})

    successes = sum(1 for r in pdf_saved_records if "saved" in r)
    print(f"\nSaved {successes}/{len(pdf_files)} PDF expense record(s) to the database.")

Found 1 PDF(s) in c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\tests\fixtures\receipts_new

Processing: utility bill.pdf
  Parsed fields: {'amount': 3301.0, 'category': 'Bills', 'description': 'Utility Bill', 'date': '2026-02-10'}
  ✓ Saved  →  id=5be7cc1e-46f7-494a-98d2-b0518fc5be1e  amount=3301.0  category=Bills  date=2026-02-10

Saved 1/1 PDF expense record(s) to the database.


## 5) Unified Multimodal Embeddings and Indexing

Generate lightweight embeddings for image-derived and document/text records and build an index with source modality metadata for traceability.

In [6]:
def _tokenize(text: str):
    return [tok for tok in ''.join(ch.lower() if ch.isalnum() else ' ' for ch in text).split() if tok]


def _jaccard_score(a: str, b: str) -> float:
    sa, sb = set(_tokenize(a)), set(_tokenize(b))
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def build_multimodal_corpus(image_items, doc_items):
    corpus = []
    for item in image_items:
        text = " ".join(str(v) for v in item["fields"].values() if v is not None)
        corpus.append({
            "modality": "image",
            "source": item["source"],
            "text": text,
            "fields": item["fields"],
        })
    for item in doc_items:
        corpus.append({
            "modality": item.get("modality", "document"),
            "source": item["source"],
            "text": item.get("text", ""),
            "fields": item.get("fields", {}),
        })
    return corpus


multimodal_corpus = build_multimodal_corpus(image_results, document_records)
print(f"Indexed records: {len(multimodal_corpus)}")
print("Sample indexed item:")
print(json.dumps(multimodal_corpus[0] if multimodal_corpus else {}, indent=2))

Indexed records: 1
Sample indexed item:
{
  "modality": "document",
  "source": "utility bill.pdf",
  "text": "UTILITY BILL\nStatement Date\nFebruary 2, 2026\nPeriod Statement from\nJanuary 1, 2026\nPeriod Statement until\nJanuary 31, 2026\nAccount No.\n12345678910\nAccount Name\nLeslie Holden\nAddress\n4344 Poco Mas Drive\nDallas, FL, 33009\nBill Summary\nPrevious Charges ($) $ 1.00\nCurrent Charges ($) $ 3,000.00\nTax ($) $ 300.00\nTotal Amount ($) $3301\nDue Date February 10, 2026\nFor any questions, please contact our billing department at billing@example.com or (123) 123-4567.\nACME Company | www.example.com\n\u00a0\nNow create your own Jotform PDF document - It's Free Create your own PDF Document\nACME Company\n123\u00a0Any Road, Suite 456\nMetropolis, NY 10001\nPhone: (123)\u00a0123-4567 | Email: billing@example.com\nWebsite: www.example.com\nREMINDERS:\n1. Present your Statement of Account when paying your utility bill.\n2. Without this document, you will be required to provide

## 6) Multimodal Search and Interactive Query Flow

Retrieve across indexed modalities, fuse top-k candidates, and produce coherent responses with source citations.

In [7]:
def multimodal_search(query: str, corpus, top_k: int = TOP_K):
    scored = []
    for row in corpus:
        score = _jaccard_score(query, row.get("text", ""))
        scored.append({**row, "score": round(score, 4)})
    ranked = sorted(scored, key=lambda x: x["score"], reverse=True)
    return ranked[:top_k]


def format_search_answer(query: str, hits):
    lines = [f"Query: {query}", "Top evidence:"]
    for idx, item in enumerate(hits, start=1):
        snippet = (item.get("text") or "")[:120].replace("\n", " ")
        lines.append(f"{idx}. [{item['modality']}] {item['source']} (score={item['score']}): {snippet}")
    return "\n".join(lines)

search_queries = [
    "coffee purchase amount and date",
    "grocery spending",
    "transportation receipt",
]

for query in search_queries:
    hits = multimodal_search(query, multimodal_corpus, top_k=TOP_K)
    print(format_search_answer(query, hits))
    print("-" * 80)

Query: coffee purchase amount and date
Top evidence:
1. [document] utility bill.pdf (score=0.0261): UTILITY BILL Statement Date February 2, 2026 Period Statement from January 1, 2026 Period Statement until January 31, 20
--------------------------------------------------------------------------------
Query: grocery spending
Top evidence:
1. [document] utility bill.pdf (score=0.0): UTILITY BILL Statement Date February 2, 2026 Period Statement from January 1, 2026 Period Statement until January 31, 20
--------------------------------------------------------------------------------
Query: transportation receipt
Top evidence:
1. [document] utility bill.pdf (score=0.0): UTILITY BILL Statement Date February 2, 2026 Period Statement from January 1, 2026 Period Statement until January 31, 20
--------------------------------------------------------------------------------


## 7) Cross-Modal Consistency and Relevance Tests

For each image-derived parse, generate an equivalent text statement, parse it through the text endpoint, and compare semantic consistency across amount/category/date.

In [8]:
def compare_modal_outputs(image_fields: dict, text_fields: dict, amount_tolerance: float = AMOUNT_TOLERANCE) -> dict:
    amount_i = image_fields.get("amount")
    amount_t = text_fields.get("amount")
    cat_i = (image_fields.get("category") or "").strip().lower()
    cat_t = (text_fields.get("category") or "").strip().lower()
    date_i = (image_fields.get("date") or "").strip()
    date_t = (text_fields.get("date") or "").strip()

    amount_match = False
    if amount_i is not None and amount_t is not None:
        try:
            amount_match = abs(float(amount_i) - float(amount_t)) <= amount_tolerance
        except Exception:
            amount_match = False

    category_match = bool(cat_i) and cat_i == cat_t
    date_match = bool(date_i) and bool(date_t) and date_i == date_t

    consistency_score = (amount_match + category_match + date_match) / 3
    return {
        "amount_match": amount_match,
        "category_match": category_match,
        "date_match": date_match,
        "consistency_score": round(consistency_score, 3),
    }


cross_modal_results = []

for item in image_results:
    try:
        image_fields = item["fields"]
        synthesized_text = image_to_text_statement(image_fields)
        text_raw = parse_expense_text(synthesized_text)
        text_fields = extract_core_fields(text_raw)
        comparison = compare_modal_outputs(image_fields, text_fields)

        result = {
            "source": item["source"],
            "synthesized_text": synthesized_text,
            "image_fields": image_fields,
            "text_fields": text_fields,
            **comparison,
        }
        cross_modal_results.append(result)
        print(f"✓ {item['source']} => score={comparison['consistency_score']}")
    except Exception as exc:
        print(f"✗ Cross-modal check failed for {item['source']}: {exc}")

if cross_modal_results:
    avg_score = statistics.mean(r["consistency_score"] for r in cross_modal_results)
    print(f"Average cross-modal consistency score: {avg_score:.3f}")
else:
    print("No cross-modal results available.")

No cross-modal results available.


## 8) Evaluation Metrics and Error Analysis

Compute retrieval quality proxies, consistency aggregates, and simple failure-mode summaries by modality/query type.

In [9]:
def recall_at_k(hits, expected_source: str, k: int = 3) -> float:
    top = hits[:k]
    return 1.0 if any(h["source"] == expected_source for h in top) else 0.0


def reciprocal_rank(hits, expected_source: str) -> float:
    for idx, h in enumerate(hits, start=1):
        if h["source"] == expected_source:
            return 1.0 / idx
    return 0.0

retrieval_scores = []
for item in image_results:
    query = f"{item['fields'].get('category', 'expense')} {item['fields'].get('amount', '')}"
    hits = multimodal_search(query, multimodal_corpus, top_k=max(5, TOP_K))
    retrieval_scores.append({
        "source": item["source"],
        "Recall@3": recall_at_k(hits, item["source"], 3),
        "MRR": reciprocal_rank(hits, item["source"]),
    })

if retrieval_scores:
    avg_recall = statistics.mean(r["Recall@3"] for r in retrieval_scores)
    avg_mrr = statistics.mean(r["MRR"] for r in retrieval_scores)
    print("Retrieval Metrics")
    print("- Avg Recall@3:", round(avg_recall, 3))
    print("- Avg MRR:", round(avg_mrr, 3))
else:
    print("No retrieval scores computed.")

if cross_modal_results:
    contradiction_rate = statistics.mean(
        1.0 if r["consistency_score"] < 0.34 else 0.0 for r in cross_modal_results
    )
    print("- Cross-modal contradiction rate:", round(contradiction_rate, 3))

print("Potential failure modes to inspect:")
print("- OCR noise on low-quality or tilted receipts")
print("- Ambiguous category labels in short descriptions")
print("- Date normalization mismatches across modalities")

No retrieval scores computed.
Potential failure modes to inspect:
- OCR noise on low-quality or tilted receipts
- Ambiguous category labels in short descriptions
- Date normalization mismatches across modalities


## 9) Modality-Specific Prompt Strategies and Templates

### Text-only parsing prompts
- Use explicit spend pattern: **amount + item + optional date**.
- Keep units normalized (`$`, ISO dates) to reduce extraction ambiguity.

### Image/OCR parsing prompts
- Expect OCR noise (`0/O`, missing decimals, broken lines).
- Favor robust cues: totals, merchant names, category hints.
- Add fallback behavior when confidence is low (ask clarifying question).

### Mixed-modality prompts
- Lead with structured facts from image parse.
- Follow with user intent (e.g., budget question/search query).
- Instruct the model to preserve source traceability and uncertainty.

### Design considerations
- **Latency**: vision+OCR is slower than text-only parsing.
- **Ambiguity handling**: normalize uncertain fields and mark low confidence.
- **Guardrails**: reject non-image files for receipt parse; enforce auth; validate categories.
- **Context budgeting**: include only top-k relevant chunks for grounded responses.

In [10]:
PROMPT_TEMPLATES = {
    "image_only": "Extract amount, category, merchant, and date from this receipt image. Return strict JSON.",
    "document_only": "Summarize the key financial facts from this parsed document text. Return bullet points.",
    "mixed_modality": "Given structured image parse + user question, answer with cited source and uncertainty notes.",
}

print("Defined templates:")
for name, template in PROMPT_TEMPLATES.items():
    print(f"- {name}: {template}")

Defined templates:
- image_only: Extract amount, category, merchant, and date from this receipt image. Return strict JSON.
- document_only: Summarize the key financial facts from this parsed document text. Return bullet points.
- mixed_modality: Given structured image parse + user question, answer with cited source and uncertainty notes.


## 10) Prompt Design Experiments and Comparison Harness

Run simple A/B prompt variants against a fixed text set and compare latency and parse success to identify modality-specific best practices.

In [11]:
ab_inputs = [
    "Spent 18.50 on coffee today",
    "I paid 62 dollars for groceries on 2026-03-09",
    "Uber ride cost me $14.20 yesterday",
]

variants = {
    "A_minimal": lambda text: text,
    "B_structured": lambda text: f"Parse as expense JSON. Input: {text}",
}

ab_results = []
for variant_name, transform in variants.items():
    for user_text in ab_inputs:
        started = time.perf_counter()
        try:
            output = parse_expense_text(transform(user_text))
            elapsed_ms = (time.perf_counter() - started) * 1000
            fields = extract_core_fields(output)
            success = fields.get("amount") is not None and fields.get("category") is not None
            ab_results.append({
                "variant": variant_name,
                "input": user_text,
                "success": bool(success),
                "latency_ms": round(elapsed_ms, 1),
            })
        except Exception as exc:
            elapsed_ms = (time.perf_counter() - started) * 1000
            ab_results.append({
                "variant": variant_name,
                "input": user_text,
                "success": False,
                "latency_ms": round(elapsed_ms, 1),
                "error": str(exc),
            })

for variant_name in variants:
    rows = [r for r in ab_results if r["variant"] == variant_name]
    success_rate = statistics.mean(1.0 if r["success"] else 0.0 for r in rows) if rows else 0.0
    avg_latency = statistics.mean(r["latency_ms"] for r in rows) if rows else 0.0
    print(f"{variant_name}: success_rate={success_rate:.2f}, avg_latency_ms={avg_latency:.1f}")

A_minimal: success_rate=1.00, avg_latency_ms=2865.4
B_structured: success_rate=1.00, avg_latency_ms=2867.8


## 11) Notebook Parameterization and Reproducible Runs

Expose configurable parameters (`API_URL`, `TOP_K`, thresholds, seed) and run a repeatable end-to-end flow with summarized artifacts.

### Validation Checklist
- [x] Image understanding flow implemented.
- [x] Document parsing flow implemented (OCR/image-first, optional PDF path).
- [x] Multimodal retrieval/interaction implemented.
- [x] Cross-modal consistency testing implemented.
- [x] Prompt strategy documentation included.

In [12]:
RUN_CONFIG = {
    "api_url": API_URL,
    "username": USERNAME,
    "top_k": TOP_K,
    "amount_tolerance": AMOUNT_TOLERANCE,
    "seed": SEED,
    "timestamp": datetime.utcnow().isoformat(),
}

artifacts = {
    "config": RUN_CONFIG,
    "image_success_count": len(image_results),
    "document_record_count": len(document_records),
    "cross_modal_count": len(cross_modal_results),
    "cross_modal_avg_score": (
        round(statistics.mean(r["consistency_score"] for r in cross_modal_results), 3)
        if cross_modal_results else None
    ),
    "ab_summary": ab_results,
}

artifact_path = PROJECT_ROOT / "docs" / "multimodal_eval_artifacts.json"
with artifact_path.open("w", encoding="utf-8") as file_obj:
    json.dump(artifacts, file_obj, indent=2)

print("Reproducible run artifacts saved:", artifact_path)
print(json.dumps({
    "image_success_count": artifacts["image_success_count"],
    "cross_modal_avg_score": artifacts["cross_modal_avg_score"],
    "ab_trials": len(ab_results)
}, indent=2))

Reproducible run artifacts saved: c:\Users\jyoth\Downloads\Project_0210\BudgetBuddy\docs\multimodal_eval_artifacts.json
{
  "image_success_count": 0,
  "cross_modal_avg_score": null,
  "ab_trials": 6
}


C:\Users\jyoth\AppData\Local\Temp\ipykernel_27968\1324272143.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
